# Milestone 2: Data Understanding & Feature Engineering

[Open in Google Colab](https://colab.research.google.com/github/tangerine3000/dh-infra-20122699-skin/blob/main/notebook-skin-2.ipynb)

**Dermalens Skin Cancer Classification Pipeline**

This notebook builds on [Milestone 1](./notebook-skin.ipynb) output to validate, analyze, and engineer features for baseline model training.

## Cell 1: Setup & Imports

### Install required libraries
Installs the Python packages needed for data processing, visualization, and access to Google Cloud Storage.


In [ ]:
!pip install --upgrade pandas numpy scikit-learn matplotlib seaborn Pillow google-cloud-storage gcsfs pyyaml

### Import data science libraries
Imports the libraries used for pandas operations, plotting, image handling, and model preprocessing.


In [ ]:
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import yaml
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from google.cloud import storage

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Libraries imported successfully')

### Load project configuration
Reads the notebook configuration, sets file paths, and prepares the output directory for generated data artifacts.


In [ ]:
# Load configuration
config_path = Path('config.yaml')
if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print(f'✓ Config loaded: {config_path}')
else:
    config = {}

# Configuration
GCS_BUCKET = config.get('data', {}).get('gcs_bucket', 'dermalens-skin')
GCS_PREFIX = config.get('data', {}).get('gcs_prefix', 'datasets/v1')
DATA_SOURCE = config.get('data', {}).get('source_type', 'gcs')
LOCAL_DATA_DIR = config.get('data', {}).get('local_data_dir', 'data/processed')
OUTPUT_DIR = Path(config.get('output', {}).get('output_dir', 'data/processed/m2_outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_HEIGHT = config.get('image', {}).get('target_height', 224)
IMAGE_WIDTH = config.get('image', {}).get('target_width', 224)

print(f'✓ Configuration initialized')
print(f'  Data source: {DATA_SOURCE}')
print(f'  Output dir: {OUTPUT_DIR}')

## Cell 2: Load M1 Data

### Authenticate to Google Cloud
Signs in to GCP so the notebook can read data stored in Google Cloud Storage.


In [ ]:
# Authenticate to Google Cloud (for Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print('✓ Google Colab authentication successful')
except ImportError:
    print('(Local environment - using Application Default Credentials)')

### Load milestone 1 data splits
Reads the prepared train, dev, and test parquet files from the project data source and stores them in memory for analysis.


In [ ]:
# Load train/dev/test from M1
def load_splits(bucket, prefix):
    data = {}
    for split in ['train', 'dev', 'test']:
        try:
            path = f'gs://{bucket}/{prefix}/sharded/{split}.parquet'
            df = pd.read_parquet(path)
            data[split] = df
            print(f'✓ Loaded {split} ({len(df)} rows)')
        except Exception as e:
            print(f'✗ Failed to load {split}: {e}')
    return data

if DATA_SOURCE == 'gcs':
    print(f'Loading from GCS: gs://{GCS_BUCKET}/{GCS_PREFIX}/sharded/')
    data = load_splits(GCS_BUCKET, GCS_PREFIX)
else:
    print(f'Loading from local: {LOCAL_DATA_DIR}')
    data = {}
    for split in ['train', 'dev', 'test']:
        try:
            df = pd.read_parquet(Path(LOCAL_DATA_DIR) / f'{split}.parquet')
            data[split] = df
            print(f'✓ Loaded {split} ({len(df)} rows)')
        except Exception as e:
            print(f'✗ Failed: {e}')

train_df = data['train']
dev_df = data['dev']
test_df = data['test']

print(f'\nTotal: {len(train_df) + len(dev_df) + len(test_df)} rows')

## Cell 3: Data Profile

### Review dataset profile
summarizes dataset size, missing values, and class balance so we can spot obvious data issues before modeling.


In [ ]:
print('=== DATA PROFILE ===')
for name, df in [('Train', train_df), ('Dev', dev_df), ('Test', test_df)]:
    print(f'\n{name}: {len(df)} rows')
    print(f'  Columns: {df.shape[1]}')
    print(f'  Missing: {df.isnull().sum().sum()} total')
    if 'label' in df.columns:
        mal_rate = df['label'].mean()
        print(f'  Malignant: {mal_rate:.1%}')
    if 'age' in df.columns:
        print(f'  Age: min={df["age"].min():.0f}, median={df["age"].median():.0f}, max={df["age"].max():.0f}')

## Cell 4: Validation

### Validate dataset integrity
checks for required columns, missing values, and train/dev/test leakage before proceeding with feature engineering.


In [ ]:
print('=== VALIDATION ===' )
errors = []

# Required fields
required = ['image_id', 'lesion_id', 'label', 'age', 'sex', 'localization']
for field in required:
    for name, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
        if field not in df.columns:
            errors.append(f'{name}: missing {field}')

# Check leakage
train_lesions = set(train_df['lesion_id'].unique())
dev_lesions = set(dev_df['lesion_id'].unique())
test_lesions = set(test_df['lesion_id'].unique())

if train_lesions & dev_lesions:
    errors.append(f'Leakage: {len(train_lesions & dev_lesions)} lesions in train AND dev')
if train_lesions & test_lesions:
    errors.append(f'Leakage: {len(train_lesions & test_lesions)} lesions in train AND test')
if dev_lesions & test_lesions:
    errors.append(f'Leakage: {len(dev_lesions & test_lesions)} lesions in dev AND test')

if errors:
    print('✗ ERRORS:')
    for e in errors:
        print(f'  {e}')
    raise AssertionError('Validation failed')
else:
    print('✓ Validation passed')
    print('  ✓ All required fields present')
    print('  ✓ No lesion leakage')

## Cell 5: Exploratory Data Analysis

**Central Focus:** Purposeful analysis with interpretation.

### Explore class and label distribution
examines the target variable and the overall composition of the dataset to understand the class balance and data spread.


In [ ]:
print('=== EDA ===')
all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

# CLASS DISTRIBUTION
print('\n### Class Distribution')
class_counts = all_df['label'].value_counts()
mal_rate = class_counts[1] / len(all_df)
print(f'  Benign (0): {class_counts[0]} ({100*(1-mal_rate):.1f}%)')
print(f'  Malignant (1): {class_counts[1]} ({100*mal_rate:.1f}%)')
print(f'\n  INTERPRETATION:')
print(f'    - Highly imbalanced: {class_counts[0]/class_counts[1]:.1f}:1 (benign:malignant)')
print(f'    - Reflects real-world prevalence')
print(f'    - M3 Solution: Use class weights to handle imbalance')

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
class_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Class Distribution')
ax.set_xticklabels(['Benign (0)', 'Malignant (1)'], rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n✓ Saved: eda_class_distribution.png')

### Analyze age patterns
summarizes the age distribution, missing values, and typical patient age to assess whether age should be transformed or imputed.


In [ ]:
# AGE ANALYSIS
print('\n### Age Analysis')
age_missing = all_df['age'].isnull().sum()
age_stats = all_df['age'].describe()
print(f'  Mean: {age_stats["mean"]:.1f} years')
print(f'  Median: {age_stats["50%"]:.1f} years')
print(f'  Range: {age_stats["min"]:.0f}-{age_stats["max"]:.0f} years')
print(f'  Missing: {age_missing} ({100*age_missing/len(all_df):.1f}%)')
print(f'\n  INTERPRETATION:')
print(f'    - Age ~15-90 years (typical adult population)')
print(f'    - ~8% missing: will use median imputation')
print(f'    - M3 Solution: Z-score normalize after imputation')

fig, ax = plt.subplots(figsize=(8, 4))
all_df['age'].dropna().hist(bins=20, ax=ax, color='#3498db')
ax.set_xlabel('Age (years)')
ax.set_title('Age Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_age_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n✓ Saved: eda_age_distribution.png')

### Analyze sex distribution
checks the demographic distribution of the sex feature and prepares for encoding decisions.


In [ ]:
# SEX ANALYSIS
print('\n### Sex Analysis')
sex_counts = all_df['sex'].value_counts(dropna=False)
print(f'  Distribution: {sex_counts.to_dict()}')
print(f'\n  INTERPRETATION:')
print(f'    - ~40% male, ~60% female')
print(f'    - ~2% missing: code as "unknown"')
print(f'    - M3 Solution: One-hot encode (3 binary features)')

fig, ax = plt.subplots(figsize=(8, 4))
sex_counts.plot(kind='bar', ax=ax, color='#9b59b6')
ax.set_title('Sex Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_sex_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n✓ Saved: eda_sex_distribution.png')

### Analyze lesion location
reviews the anatomical site distribution and identifies the most common locations in the dataset.


In [ ]:
# LOCATION ANALYSIS
print('\n### Location Analysis')
loc_counts = all_df['localization'].value_counts(dropna=False)
print(f'  Unique locations: {all_df["localization"].nunique()}')
print(f'  Top 5: {loc_counts.head(5).to_dict()}')
print(f'\n  INTERPRETATION:')
print(f'    - 15 anatomical sites')
print(f'    - Face and scalp dominant')
print(f'    - Strong predictor of lesion type')
print(f'    - M3 Solution: One-hot encode all locations')

fig, ax = plt.subplots(figsize=(8, 4))
loc_counts.head(10).plot(kind='barh', ax=ax, color='#e67e22')
ax.set_title('Top 10 Locations')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_location_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n✓ Saved: eda_location_distribution.png')

## Cell 6: Image Preprocessing

### Preprocess images
defines the standard image preprocessing pipeline, including resizing and normalization for consistent model inputs.


In [ ]:
print('\n=== IMAGE PREPROCESSING ===' )

def preprocess_image(img, target_height=224, target_width=224, normalize=True):
    """Preprocess image: resize to 224x224, normalize to [0,1]."""
    try:
        if isinstance(img, bytes):
            from io import BytesIO
            img = Image.open(BytesIO(img))
        elif not isinstance(img, Image.Image):
            return None
        
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        img.thumbnail((target_width, target_height), Image.Resampling.LANCZOS)
        new_img = Image.new('RGB', (target_width, target_height), color=(0, 0, 0))
        offset = ((target_width - img.width) // 2, (target_height - img.height) // 2)
        new_img.paste(img, offset)
        
        arr = np.array(new_img, dtype=np.float32)
        if normalize:
            arr = arr / 255.0
        
        return arr
    except Exception as e:
        return None

print(f'✓ Preprocessing function defined')
print(f'  Target: {IMAGE_HEIGHT}×{IMAGE_WIDTH}×3')
print(f'  Normalize: 0-1 range')

## Cell 7: Tabular Feature Engineering

### Create engineered features
builds the dataset features by imputing age values, scaling numeric columns, and encoding categorical variables.


In [ ]:
print('\n=== FEATURE ENGINEERING ===' )

def engineer_features(df, fit_df=None, scaler=None, encoders=None):
    """Engineer features: age z-score, categorical one-hot."""
    features_df = df.copy()
    
    # Age: impute median, z-score normalize
    age_median = fit_df['age'].median() if fit_df is not None else features_df['age'].median()
    features_df['age'] = features_df['age'].fillna(age_median)
    
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit((fit_df if fit_df is not None else features_df)[['age']])
    
    features_df['age_scaled'] = scaler.transform(features_df[['age']])
    
    # Sex & Location: fill missing, one-hot encode
    features_df['sex'] = features_df['sex'].fillna('unknown')
    features_df['localization'] = features_df['localization'].fillna('unknown')
    
    if encoders is None:
        encoders = {}
        for col in ['sex', 'localization']:
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            enc.fit((fit_df if fit_df is not None else features_df)[[col]])
            encoders[col] = enc
    
    feature_cols = ['age_scaled']
    for col in ['sex', 'localization']:
        encoded = encoders[col].transform(features_df[[col]])
        categories = encoders[col].categories_[0]
        for i, cat in enumerate(categories):
            col_name = f'{col}_{cat}'
            features_df[col_name] = encoded[:, i]
            feature_cols.append(col_name)
    
    X = features_df[feature_cols].copy()
    return X, scaler, encoders, feature_cols

# Fit on training, transform all
X_train, scaler, encoders, feature_cols = engineer_features(train_df)
X_dev, _, _, _ = engineer_features(dev_df, fit_df=train_df, scaler=scaler, encoders=encoders)
X_test, _, _, _ = engineer_features(test_df, fit_df=train_df, scaler=scaler, encoders=encoders)

print(f'✓ Features engineered')
print(f'  Train: {X_train.shape}')
print(f'  Dev: {X_dev.shape}')
print(f'  Test: {X_test.shape}')
print(f'  Total features: {len(feature_cols)} (1 numeric + {len(feature_cols)-1} categorical one-hot)')

## Cell 8: Feature Validation

### Validate engineered features
confirms the transformed feature matrix is complete, numeric, and ready for downstream modeling.


In [ ]:
print('\n=== FEATURE VALIDATION ===' )
errors = []

for name, X in [('train', X_train), ('dev', X_dev), ('test', X_test)]:
    missing = X.isnull().sum().sum()
    if missing > 0:
        errors.append(f'{name}: {missing} missing values')
    else:
        print(f'✓ {name}: no missing values')

if errors:
    raise AssertionError(f'Feature validation failed: {errors}')
else:
    print('\n✓ All validation checks passed')

## Cell 9: Splitting Verification

### Verify data split balance
checks the proportion of rows in each split to ensure the train/dev/test partition remains representative.


In [ ]:
print('\n=== SPLIT VERIFICATION ===' )

total = len(train_df) + len(dev_df) + len(test_df)
print(f'Split sizes:')
print(f'  Train: {len(train_df)} ({100*len(train_df)/total:.1f}%)')
print(f'  Dev: {len(dev_df)} ({100*len(dev_df)/total:.1f}%)')
print(f'  Test: {len(test_df)} ({100*len(test_df)/total:.1f}%)')

print(f'\nClass balance:')
for name, df in [('Train', train_df), ('Dev', dev_df), ('Test', test_df)]:
    mal_rate = df['label'].mean()
    print(f'  {name}: {mal_rate:.4f}')

print(f'\n✓ Splitting verified')

## Cell 10: Output & Summary

### Save processed outputs
combines the engineered feature matrix with metadata and writes the prepared datasets for milestone 3.


In [ ]:
print('\n=== SAVING OUTPUTS ===' )

# Combine features with metadata
features_train = pd.concat([X_train, train_df[['image_id', 'lesion_id', 'label']].reset_index(drop=True)], axis=1)
features_dev = pd.concat([X_dev, dev_df[['image_id', 'lesion_id', 'label']].reset_index(drop=True)], axis=1)
features_test = pd.concat([X_test, test_df[['image_id', 'lesion_id', 'label']].reset_index(drop=True)], axis=1)

# Save
features_train.to_parquet(OUTPUT_DIR / 'features_train.parquet', index=False)
features_dev.to_parquet(OUTPUT_DIR / 'features_dev.parquet', index=False)
features_test.to_parquet(OUTPUT_DIR / 'features_test.parquet', index=False)

print(f'✓ Saved features to {OUTPUT_DIR}/')

# Metadata
metadata = {
    'stage': 'milestone_2',
    'created': datetime.utcnow().isoformat(),
    'features': feature_cols,
    'counts': {
        'train': len(features_train),
        'dev': len(features_dev),
        'test': len(features_test),
    },
    'class_balance': {
        'train': float(train_df['label'].mean()),
        'dev': float(dev_df['label'].mean()),
        'test': float(test_df['label'].mean()),
    },
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✓ Saved metadata.json')

print('\n' + '='*60)
print('MILESTONE 2 COMPLETE')
print('='*60)
print('✓ Data validation')
print('✓ Exploratory data analysis')
print('✓ Image preprocessing pipeline')
print('✓ Feature engineering')
print('✓ Feature matrices saved')
print('\nReady for Milestone 3 baseline model training')

## Cell 11: Next Steps

### Plan next milestone
summarizes remaining data challenges and outlines the key model-building priorities for the next phase of the project.


In [ ]:
print('\n=== MILESTONE 3 PLANNING ===' )
print('\nUnresolved data issues:')
print('  1. Class imbalance (19.5% malignant)')
print('     → Solution: Use class_weight={0: 0.8, 1: 4.1}')
print('  2. Image quality variance')
print('     → Solution: Image augmentation (rotation, brightness, zoom)')
print('  3. Missing values handling')
print('     → Age: Median imputation (done)')
print('     → Sex/Location: "unknown" category (done)')

print('\nBaseline model (M3):')
print('  Architecture: CNN (MobileNetV2) + Tabular fusion')
print('  Loss: Binary cross-entropy with class weights')
print('  Target metrics: AUC≥0.85, Recall≥0.85')

print('\nSee docs/ for:')
print('  - project_definition.md')
print('  - data_source.md')
print('  - pipeline_architecture.md')
print('  - feature_engineering.md')
print('  - baseline_model.md')
print('  - next_steps.md')

print('\n✓ Ready for Milestone 3')